# D2-to-D3 TDI/IAG CLR heatmap for selected D3 programs

This notebook recreates the three-row heatmap using:

- **Control / Control** as a zero reference row;
- **TDI**: `D2_LM+TDI->D3_AE+TDI` vs matched `D3_AE` control;
- **IAG**: `D2_LM+IAG->D3_AE+IAG` vs matched `D3_AE` control.

Program columns are zero-based and ordered as:

```text
P2, P3, P8, P11, P33, P34
```

Labels are taken from the first annotation table in *Program annotation notes (Kyle+Hunter).pdf*:

- P2 - Artery endothelium 1
- P3 - Endothelial function 1
- P8 - Artery endothelium 2
- P11 - Endothelial function 2
- P33 - Anterior-like mesoderm
- P34 - Mesenchyme, YAP/TAZ-induced

CLR is applied at the **cell level across all 50 projected programs**, then averaged per biological sample. Treatment effects and statistics are calculated from matched biological-replicate sample means.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",  # keep text editable in Illustrator
})

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------
BASE = Path("/scratch/users/edtng/20260325_Hippo_timecourse")
PROJ_DIR = BASE / "perturbseq_cNMF_projection"
H5AD_PATH = PROJ_DIR / "Hippo_timecourse_combined.projected_perturbseq_cNMF_50_dt_0_2.zero_based.h5ad"

OUT_DIR = PROJ_DIR / "D2toD3_CLR_heatmap_P2_P3_P8_P11_P33_P34"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCORE_KEY = "X_perturbseq_cNMF_usage"
SAMPLE_COL = "sample_name"

# ------------------------------------------------------------------
# CLR settings
# ------------------------------------------------------------------
# programDE-style structure: floor -> renormalize -> log -> center.
CLR_EPS = 1e-6
CLR_LOG_BASE = 2

# ------------------------------------------------------------------
# Requested programs and PDF-derived annotation labels
# ------------------------------------------------------------------
SELECTED_PROGRAMS = ["P2", "P3", "P8", "P11", "P33", "P34"]
PROGRAM_LABELS = {
    "P2": "P2\nArtery endothelium 1",
    "P3": "P3\nEndothelial function 1",
    "P8": "P8\nArtery endothelium 2",
    "P11": "P11\nEndothelial function 2",
    "P33": "P33\nAnterior-like mesoderm",
    "P34": "P34\nMesenchyme, YAP/TAZ-induced",
}

CONTROL_CONDITION = "D3_AE"
TREATED_CONDITIONS = {
    "TDI": "D2_LM+TDI->D3_AE+TDI",
    "IAG": "D2_LM+IAG->D3_AE+IAG",
}
ROW_ORDER = ["Control", "TDI", "IAG"]

# ------------------------------------------------------------------
# Statistical display
# ------------------------------------------------------------------
# Paired t-test across matched biological replicate means.
# Stars use BH-adjusted q-values across 2 treatments x 6 programs.
STAR_BY = "padj_bh"  # alternatives: "pvalue" or None

print("H5AD_PATH:", H5AD_PATH)
print("exists:", H5AD_PATH.exists())
print("OUT_DIR:", OUT_DIR)
print("programs:", SELECTED_PROGRAMS)

## QC cutoffs and helper functions

In [ ]:
# Same condition-specific QC cutoffs used in the earlier time-course analysis.
cond_pct_ribo_cutoffs = {
    'D0_hPSC': 2.4,
    'D1_MPS': 2.3,
    'D1_MPS+TDI': 2.5,
    'D1_MPS+IAG': 2.2,
    'D1.33_LM': 1.9,
    'D1.33_LM+TDI': 2.1,
    'D1.33_LM+IAG': 2.2,
    'D1.67_LM': 2.0,
    'D1.67_LM+TDI': 2.2,
    'D1.67_LM+IAG': 1.9,
    'D2_LM': 2.0,
    'D2_LM+TDI': 2.4,
    'D2_LM+IAG': 1.9,
    'D3_AE': 1.0,
    'D3_AE+TDI': 2.0,
    'D3_AE+IAG': 1.0,
    'D2_LM+TDI->D3_AE+TDI': 2.5,
    'D2_LM+IAG->D3_AE+IAG': 1.0,
}

cond_pct_mt_cutoffs = {
    'D0_hPSC': 5.5,
    'D1_MPS': 3.0,
    'D1_MPS+TDI': 3.0,
    'D1_MPS+IAG': 4.0,
    'D1.33_LM': 3.5,
    'D1.33_LM+TDI': 3.75,
    'D1.33_LM+IAG': 5.0,
    'D1.67_LM': 4.0,
    'D1.67_LM+TDI': 5.5,
    'D1.67_LM+IAG': 3.5,
    'D2_LM': 6.5,
    'D2_LM+TDI': 8.0,
    'D2_LM+IAG': 3.5,
    'D3_AE': 3.25,
    'D3_AE+TDI': 7.0,
    'D3_AE+IAG': 3.75,
    'D2_LM+TDI->D3_AE+TDI': 7.0,
    'D2_LM+IAG->D3_AE+IAG': 3.75,
}


def parse_sample_name(sample_name):
    """Return (condition, replicate_number) from a sample such as D3_AE_rep2."""
    sample_name = str(sample_name)
    match = re.match(r"^(.*)_rep(\d+)$", sample_name)
    if match is None:
        raise ValueError(f"Could not parse sample_name: {sample_name!r}")
    return match.group(1), int(match.group(2))


def clr_transform_rows(compositions, eps=1e-6, log_base=2):
    """CLR transform cells x programs after flooring and row renormalization."""
    x = np.asarray(compositions, dtype=np.float64)
    if x.ndim != 2:
        raise ValueError(f"Expected a 2D usage matrix, got shape {x.shape}")
    if np.any(x < 0):
        raise ValueError("CLR input contains negative values.")

    row_sums = x.sum(axis=1, keepdims=True)
    x = np.divide(x, row_sums, out=np.zeros_like(x), where=row_sums > 0)
    x = np.maximum(x, eps)
    x = x / x.sum(axis=1, keepdims=True)

    if log_base == 2:
        log_x = np.log2(x)
    elif log_base in (None, "natural", "e"):
        log_x = np.log(x)
    else:
        raise ValueError("log_base must be 2 or 'natural'.")

    clr = log_x - log_x.mean(axis=1, keepdims=True)
    return clr.astype(np.float32)


def bh_adjust(pvalues):
    """Benjamini-Hochberg correction, preserving NaN positions."""
    pvalues = np.asarray(pvalues, dtype=float)
    out = np.full(pvalues.shape, np.nan, dtype=float)
    valid = np.isfinite(pvalues)
    p = pvalues[valid]
    if len(p) == 0:
        return out
    order = np.argsort(p)
    ranked = p[order]
    n = len(ranked)
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    back = np.empty_like(adjusted)
    back[order] = adjusted
    out[valid] = back
    return out


def significance_stars(value):
    if value is None or not np.isfinite(value):
        return ""
    if value < 0.001:
        return "***"
    if value < 0.01:
        return "**"
    if value < 0.05:
        return "*"
    return ""

## Load projected usage, apply QC, and compute sample-level CLR means

The large RNA expression matrix stays backed on disk. Only `obs` and the 50-column projected usage matrix are loaded into memory.

In [ ]:
adata = sc.read_h5ad(H5AD_PATH, backed="r")
obs = adata.obs.copy()
usage = np.asarray(adata.obsm[SCORE_KEY])

projection_meta = adata.uns.get("perturbseq_cNMF_projection", {})
program_names = [str(x) for x in projection_meta.get("program_names", [])]
if len(program_names) != usage.shape[1]:
    program_names = [f"P{i}" for i in range(usage.shape[1])]

print("usage shape:", usage.shape)
print("first/last programs:", program_names[:5], program_names[-5:])

missing_programs = [p for p in SELECTED_PROGRAMS if p not in program_names]
if missing_programs:
    raise ValueError(f"Selected programs missing from saved projection: {missing_programs}")

if SAMPLE_COL not in obs.columns:
    raise KeyError(f"{SAMPLE_COL!r} not found in adata.obs")
for col in ["pct_counts_mt", "pct_counts_ribo"]:
    if col not in obs.columns:
        raise KeyError(f"{col!r} not found in adata.obs")

parsed = obs[SAMPLE_COL].astype(str).map(parse_sample_name)
obs["condition_for_heatmap"] = [x[0] for x in parsed]
obs["replicate_for_heatmap"] = [x[1] for x in parsed]
obs["pct_counts_mt"] = pd.to_numeric(obs["pct_counts_mt"], errors="coerce")
obs["pct_counts_ribo"] = pd.to_numeric(obs["pct_counts_ribo"], errors="coerce")

# Keep only the three conditions needed for this heatmap before CLR aggregation.
needed_conditions = {CONTROL_CONDITION, *TREATED_CONDITIONS.values()}
condition_mask = obs["condition_for_heatmap"].isin(needed_conditions).to_numpy()

qc_keep = np.zeros(obs.shape[0], dtype=bool)
qc_records = []
for condition in needed_conditions:
    cond_mask = (obs["condition_for_heatmap"].to_numpy() == condition)
    fail_mt = cond_mask & (obs["pct_counts_mt"].to_numpy() >= cond_pct_mt_cutoffs[condition])
    fail_ribo = cond_mask & (obs["pct_counts_ribo"].to_numpy() <= cond_pct_ribo_cutoffs[condition])
    fail_any = fail_mt | fail_ribo
    keep_cond = cond_mask & ~fail_any
    qc_keep |= keep_cond
    qc_records.append({
        "condition": condition,
        "n_before": int(cond_mask.sum()),
        "n_fail_mt": int(fail_mt.sum()),
        "n_fail_low_ribo": int(fail_ribo.sum()),
        "n_removed_any": int(fail_any.sum()),
        "n_after": int(keep_cond.sum()),
        "pct_counts_mt_cutoff": cond_pct_mt_cutoffs[condition],
        "pct_counts_ribo_cutoff": cond_pct_ribo_cutoffs[condition],
    })

qc_summary = pd.DataFrame(qc_records).sort_values("condition")
qc_summary.to_csv(OUT_DIR / "QC_summary_selected_D2toD3_conditions.tsv", sep="\t", index=False)
display(qc_summary)

obs_f = obs.loc[qc_keep].copy()
usage_f = usage[qc_keep, :]
print(f"Cells retained for heatmap: {usage_f.shape[0]:,}")

# Cell-level CLR across all projected programs.
clr_f = clr_transform_rows(usage_f, eps=CLR_EPS, log_base=CLR_LOG_BASE)
print("CLR shape:", clr_f.shape)
print("Max absolute row mean after CLR:", float(np.max(np.abs(clr_f.mean(axis=1)))))

clr_df = pd.DataFrame(clr_f, index=obs_f.index.astype(str), columns=program_names)
clr_df[SAMPLE_COL] = obs_f[SAMPLE_COL].astype(str).to_numpy()
clr_df["condition"] = obs_f["condition_for_heatmap"].astype(str).to_numpy()
clr_df["replicate"] = obs_f["replicate_for_heatmap"].astype(int).to_numpy()

sample_clr = clr_df.groupby(SAMPLE_COL, observed=True)[program_names].mean()
sample_meta = (
    clr_df[[SAMPLE_COL, "condition", "replicate"]]
    .drop_duplicates(SAMPLE_COL)
    .set_index(SAMPLE_COL)
    .loc[sample_clr.index]
)

sample_clr.to_csv(OUT_DIR / "sample_mean_log2CLR_projected_program_usage.tsv", sep="\t")
sample_meta.to_csv(OUT_DIR / "sample_metadata_for_CLR_heatmap.tsv", sep="\t")

display(sample_meta.sort_values(["condition", "replicate"]))

adata.file.close()

## Matched-replicate effects and paired statistical tests

For each treatment and program:

```text
rep1 effect = treated_rep1 CLR - control_rep1 CLR
rep2 effect = treated_rep2 CLR - control_rep2 CLR
rep3 effect = treated_rep3 CLR - control_rep3 CLR
```

A two-sided paired t-test is applied to the matched sample-level CLR values. Benjamini-Hochberg correction is applied across all 12 treatment-program tests.

In [ ]:
effect_records = []
paired_records = []

control_meta = sample_meta[sample_meta["condition"] == CONTROL_CONDITION]
control_by_rep = control_meta.reset_index().set_index("replicate")[SAMPLE_COL].to_dict()

for treatment, treated_condition in TREATED_CONDITIONS.items():
    treated_meta = sample_meta[sample_meta["condition"] == treated_condition]
    treated_by_rep = treated_meta.reset_index().set_index("replicate")[SAMPLE_COL].to_dict()
    matched_reps = sorted(set(control_by_rep).intersection(treated_by_rep))
    print(treatment, treated_condition, "matched reps:", matched_reps)

    if len(matched_reps) < 2:
        raise ValueError(f"Fewer than two matched replicates for {treatment}: {matched_reps}")

    for program in SELECTED_PROGRAMS:
        control_values = np.array([sample_clr.loc[control_by_rep[r], program] for r in matched_reps], dtype=float)
        treated_values = np.array([sample_clr.loc[treated_by_rep[r], program] for r in matched_reps], dtype=float)
        paired_diff = treated_values - control_values

        t_stat, pvalue = stats.ttest_rel(treated_values, control_values, nan_policy="omit")

        effect_records.append({
            "treatment": treatment,
            "treated_condition": treated_condition,
            "control_condition": CONTROL_CONDITION,
            "program": program,
            "program_label": PROGRAM_LABELS[program].replace("\n", " - "),
            "n_matched_pairs": len(matched_reps),
            "mean_control_clr": float(control_values.mean()),
            "mean_treated_clr": float(treated_values.mean()),
            "mean_diff_clr": float(paired_diff.mean()),
            "sd_diff_clr": float(paired_diff.std(ddof=1)) if len(paired_diff) > 1 else np.nan,
            "t_statistic": float(t_stat),
            "pvalue": float(pvalue),
            "n_positive_diffs": int((paired_diff > 0).sum()),
            "n_negative_diffs": int((paired_diff < 0).sum()),
        })

        for rep, control_value, treated_value, diff in zip(matched_reps, control_values, treated_values, paired_diff):
            paired_records.append({
                "treatment": treatment,
                "treated_condition": treated_condition,
                "control_condition": CONTROL_CONDITION,
                "program": program,
                "replicate": rep,
                "control_sample": control_by_rep[rep],
                "treated_sample": treated_by_rep[rep],
                "control_clr": control_value,
                "treated_clr": treated_value,
                "paired_diff_clr": diff,
            })

effects = pd.DataFrame(effect_records)
effects["padj_bh"] = bh_adjust(effects["pvalue"].to_numpy())
paired_values = pd.DataFrame(paired_records)

effects.to_csv(OUT_DIR / "D2toD3_CLR_selected_program_effects_and_statistics.tsv", sep="\t", index=False)
paired_values.to_csv(OUT_DIR / "D2toD3_CLR_selected_program_paired_replicate_values.tsv", sep="\t", index=False)

display(effects.sort_values(["treatment", "program"]))

## Build and plot the square-cell heatmap

The control row is exactly zero and is a visual reference only. No statistical test is assigned to control/control.

In [ ]:
heatmap_values = pd.DataFrame(0.0, index=ROW_ORDER, columns=SELECTED_PROGRAMS)
heatmap_pvalues = pd.DataFrame(np.nan, index=ROW_ORDER, columns=SELECTED_PROGRAMS)
heatmap_qvalues = pd.DataFrame(np.nan, index=ROW_ORDER, columns=SELECTED_PROGRAMS)

for _, row in effects.iterrows():
    heatmap_values.loc[row["treatment"], row["program"]] = row["mean_diff_clr"]
    heatmap_pvalues.loc[row["treatment"], row["program"]] = row["pvalue"]
    heatmap_qvalues.loc[row["treatment"], row["program"]] = row["padj_bh"]

heatmap_values.to_csv(OUT_DIR / "D2toD3_CLR_heatmap_values.tsv", sep="\t")
heatmap_pvalues.to_csv(OUT_DIR / "D2toD3_CLR_heatmap_pvalues.tsv", sep="\t")
heatmap_qvalues.to_csv(OUT_DIR / "D2toD3_CLR_heatmap_qvalues.tsv", sep="\t")

display(heatmap_values)

In [ ]:
vals = heatmap_values.to_numpy(dtype=float)
vmax = float(np.nanmax(np.abs(vals)))
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1e-9

n_rows, n_cols = heatmap_values.shape
# Square data cells: aspect='equal'; figure dimensions leave room for long labels and colorbar.
fig, ax = plt.subplots(figsize=(10.2, 4.8))
mesh = ax.pcolormesh(
    np.arange(n_cols + 1),
    np.arange(n_rows + 1),
    vals,
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
    shading="flat",
    edgecolors="white",
    linewidth=1.2,
)
ax.set_aspect("equal")
ax.set_xlim(0, n_cols)
ax.set_ylim(n_rows, 0)
ax.set_xticks(np.arange(n_cols) + 0.5)
ax.set_yticks(np.arange(n_rows) + 0.5)
ax.set_xticklabels([PROGRAM_LABELS[p] for p in SELECTED_PROGRAMS], rotation=48, ha="right", fontsize=9)
ax.set_yticklabels(ROW_ORDER, fontsize=12)
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title("D2→D3 treatment effects on selected D3 programs", fontsize=15, weight="bold", pad=12)
ax.set_xlabel("Projected perturb-seq cNMF program", labelpad=8)
ax.set_ylabel("Small molecule")

for i, treatment in enumerate(ROW_ORDER):
    for j, program in enumerate(SELECTED_PROGRAMS):
        value = heatmap_values.loc[treatment, program]
        if treatment == "Control":
            txt = "0.00"
        else:
            if STAR_BY == "padj_bh":
                stat_value = heatmap_qvalues.loc[treatment, program]
            elif STAR_BY == "pvalue":
                stat_value = heatmap_pvalues.loc[treatment, program]
            else:
                stat_value = np.nan
            txt = f"{value:+.2f}{significance_stars(stat_value)}"

        text_color = "white" if abs(value) > 0.62 * vmax else "black"
        ax.text(j + 0.5, i + 0.5, txt, ha="center", va="center", fontsize=10, color=text_color)

cbar = fig.colorbar(mesh, ax=ax, pad=0.025, fraction=0.045)
cbar.set_label("Δ mean log2 CLR\n(treated − matched D3_AE control)", fontsize=10)

fig.subplots_adjust(left=0.11, right=0.89, bottom=0.33, top=0.86)

stem = "D2toD3_TDI_IAG_CLR_heatmap_P2_P3_P8_P11_P33_P34_square_cells"
for ext in ["svg", "pdf", "png"]:
    fig.savefig(OUT_DIR / f"{stem}.{ext}", bbox_inches="tight", dpi=300 if ext == "png" else None)
plt.show()
plt.close(fig)

print("Saved figure and tables to:", OUT_DIR)